In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, current_date, lit
from pyspark.sql.utils import AnalysisException

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.raw")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.gold")

spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.raw.arquivos_tc2")



In [0]:

#Função para criar as tabelas delta na camada bronze
def carregar_bronze(
    csv_path: str,
    catalogo: str,
    schema: str,
    tabela: str,
    sep: str = ",",
    encoding: str = "UTF-8",
    header: bool = True,
    infer_schema: bool = True
):
    #Iniciando 
    print(f"Ingestão Raw Bronze: {tabela}")
  
#leitura do arquivo com try catch pra pegar possíveis erros
    try:

        df = (
            spark.read
            .option("header", header)
            .option("inferSchema", infer_schema)
            .option("sep", sep)
            .option("encoding", encoding)
            .option("mode", "FAILFAST")
            .csv(csv_path)
        )

    except Exception as e:
        raise Exception(f"Erro ao ler CSV:\n{e}")

    #verifica se o arquivo está vazio
    if df.isEmpty():
        raise Exception("Arquivo vazio.")
    
    #aqui vejo se o arquivo tem uma coluna só (problema de delimitador errado)
    if len(df.columns) <= 1:
        raise Exception(
            "Provável delimitador incorreto."
        )

    #verifica se tem colunas com dois nomes. Decidi fazer aqui pq pode dar problema na criação da tabela na bronze em vez de fazer na silver
    if len(df.columns) != len(set(df.columns)):
        raise Exception(
            "Existem colunas duplicadas."
        )

    #Adiciono timestamp, data_ref e o nome do arquivo origem 
    df = (
        df
        .withColumn("dt_ingestao", current_timestamp())
        .withColumn("dh_referencia_ingestao", current_date())
        .withColumn("arquivo_origem", lit(csv_path))
    )

    nome_tabela = f"{catalogo}.{schema}.{tabela}"

    #coloquei o append pra manter o histórico particionado
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(nome_tabela)
    )

    #printo no console para ver se deu tudo certo
    print(f"Carga ingerida: {nome_tabela}")
    print(f"Linhas carregadas: {df.count():,}")

    print("Carga Bronze finalizada.")
    print("")


In [0]:

#Definindo o caminho dos arquivos no volume do schema RAW
path_meta_brasil = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv"
path_meta_municipio = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv"
path_meta_uf = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv"
path_avaliacao_municipio = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_municipio.csv"
path_avaliacao_uf = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_uf.csv"
path_avaliacao_alunos = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_alunos.csv"
path_tab_municipios_br = "/Volumes/workspace/raw/arquivos_tc2/BRASIL_2024_MUNICIPIOS.csv"

#Chamada das Funções para a criação das Delta Tables na bronze
carregar_bronze(
    csv_path = path_meta_brasil,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_meta_alfabetizacao_brasil"
)

carregar_bronze(
    csv_path = path_meta_municipio,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_meta_alfabetizacao_municipio"
)

carregar_bronze(
    csv_path = path_meta_uf,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_meta_alfabetizacao_uf"
)

carregar_bronze(
    csv_path = path_avaliacao_municipio,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_avaliacao_municipio"
)

carregar_bronze(
    csv_path = path_avaliacao_uf,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_avaliacao_uf"
)

carregar_bronze(
    csv_path = path_avaliacao_alunos,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_avaliacao_alunos"
)

#esse arquivo é diferente o encoding e a separação do csv
carregar_bronze(
    csv_path = path_tab_municipios_br,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_municipios_br",
    sep=";",
    encoding="ISO-8859-1"
)

In [0]:
#Tratamento da tabela de municípios

from pyspark.sql import functions as F
from pyspark.sql import Window

#definição das tabelas para facilitar os próximos tratamentos
BRONZE_TABELA = f"workspace.bronze.b_municipios_br"
SILVER_TABELA = f"workspace.silver.s_municipios_br"

df_bronze = spark.table(BRONZE_TABELA)

#trazer a carga mais recente
dh_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dh_max"))
    .collect()[0]["dh_max"]
)
 
df = df_bronze.filter(F.col("dh_referencia_ingestao") == dh_mais_recente)
 
print(f"[s_municipios_br] dh_referencia_ingestao utilizada: {dh_mais_recente}")
print(f"[s_municipios_br] linhas na carga mais recente: {df.count()}")
 
#Aqui realizo um tratamento para padronizar os nomes das colunas caso venham com espaço
df = df.select(
    F.col("cd_uf").cast("int").alias("cd_uf"),
    F.trim(F.col("nm_uf")).alias("nm_uf"),
    F.col("cd_reg_geog_intermed").cast("int").alias("cd_reg_geog_intermediaria"),
    F.trim(F.col("nm_reg_geog_intermed")).alias("nm_reg_geog_intermediaria"),
    F.col("cd_reg_geog_imediata").cast("int").alias("cd_reg_geog_imediata"),
    F.trim(F.col("nm_reg_geog_imediata")).alias("nm_reg_geog_imediata"),
    F.col("cd_municipio_completo").alias("cd_municipio_completo_raw"),
    F.trim(F.col("nm_municipio")).alias("nm_municipio"),
)

#aqui vou manter o id_municipio como string com zeros a esquerda. Caso ocorra alguma modificação.
df = df.withColumn(
    "id_municipio",
    F.lpad(F.col("cd_municipio_completo_raw").cast("string"), 7, "0")
).drop("cd_municipio_completo_raw")

#aqui adiciono uma coluna com os nomes de cada UF
mapa_sigla_uf = {
    "Rondônia": "RO", "Acre": "AC", "Amazonas": "AM", "Roraima": "RR",
    "Pará": "PA", "Amapá": "AP", "Tocantins": "TO", "Maranhão": "MA",
    "Piauí": "PI", "Ceará": "CE", "Rio Grande do Norte": "RN", "Paraíba": "PB",
    "Pernambuco": "PE", "Alagoas": "AL", "Sergipe": "SE", "Bahia": "BA",
    "Minas Gerais": "MG", "Espírito Santo": "ES", "Rio de Janeiro": "RJ",
    "São Paulo": "SP", "Paraná": "PR", "Santa Catarina": "SC",
    "Rio Grande do Sul": "RS", "Mato Grosso do Sul": "MS", "Mato Grosso": "MT",
    "Goiás": "GO", "Distrito Federal": "DF",
}
mapa_expr = F.create_map([F.lit(x) for kv in mapa_sigla_uf.items() for x in kv])
df = df.withColumn("sigla_uf", mapa_expr[F.col("nm_uf")])

#Verificação de duplicadas
total_linhas = df.count()
total_chaves_distintas = df.select("id_municipio").distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_municipios_br] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 
if qtd_duplicados > 0:
    #Em caso de problemas, aqui deixa a última carga
    janela = Window.partitionBy("id_municipio").orderBy(F.col("nm_municipio").desc())
    df = (
        df.withColumn("_rn", F.row_number().over(janela))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )

#Verificação de nulos
qtd_id_nulo = df.filter(F.col("id_municipio").isNull() | (F.col("id_municipio") == "")).count()
qtd_sigla_nula = df.filter(F.col("sigla_uf").isNull()).count()
 
print(f"[s_municipios_br] id_municipio nulo/vazio: {qtd_id_nulo} | sigla_uf não mapeada: {qtd_sigla_nula}")

assert qtd_id_nulo == 0, "Encontrados municípios com id_municipio nulo/vazio — investigar a Bronze antes de prosseguir."
#assert qtd_sigla_nula == 0, "Encontrado nm_uf sem correspondência no mapa de siglas — atualizar o de-para."

#criacao da coluna de referencia da ingestao
df = (
    df.withColumn("dh_referencia_ingestao_origem", F.lit(dh_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)

(
    df.select(
        "id_municipio", "nm_municipio",
        "cd_uf", "sigla_uf", "nm_uf",
        "cd_reg_geog_intermediaria", "nm_reg_geog_intermediaria",
        "cd_reg_geog_imediata", "nm_reg_geog_imediata",
        "dh_referencia_ingestao_origem", "dh_processamento_silver",
    )
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_municipios_br] escrita concluída em {SILVER_TABELA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
#Tratamento da tabela de alunos
BRONZE_TABELA = f"workspace.bronze.b_avaliacao_alunos"
SILVER_TABELA = f"workspace.silver.s_avaliacao_alunos"
MUNICIPIOS_TABELA = f"workspace.silver.s_municipios_br"

#trazer carga mais recente da bronze
df_bronze = spark.table(BRONZE_TABELA)
 
dh_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dh_max"))
    .collect()[0]["dh_max"]
)
 
df = df_bronze.filter(F.col("dh_referencia_ingestao") == dh_mais_recente)
 
print(f"[s_avaliacao_alunos] dh_referencia_ingestao utilizada: {dh_mais_recente}")
print(f"[s_avaliacao_alunos] linhas na carga mais recente: {df.count()}")

#aqui padronizei o nome das colunas. Todas que tem id (id_municipio, id_escola, id_aluno) mantive com o zero a esquerda

df = df.select(
    F.col("ano").cast("int").alias("ano"),
    F.lpad(F.col("id_municipio").cast("string"), 7, "0").alias("id_municipio"),
    F.col("id_escola").cast("string").alias("id_escola"),
    F.col("id_aluno").cast("string").alias("id_aluno"),
    F.col("caderno").cast("int").alias("caderno"),
    F.col("serie").cast("int").alias("serie"),
    F.col("rede").cast("int").alias("cd_rede"),
    F.col("presenca").cast("int").alias("presenca"),
    F.col("preenchimento_caderno").cast("int").alias("preenchimento_caderno"),
    F.col("alfabetizado").cast("int").alias("alfabetizado"),
    F.col("proficiencia").cast("double").alias("proficiencia"),
    F.col("peso_aluno").cast("double").alias("peso_aluno"),
)
 
#aqui criei a flag de avalização valida pois há casos que o aluno não tem presença ou não tem o caderno preenchido, portanto vem vazio.
#vi isso na parte de exploração de dados e pontuei lá
df = df.withColumn(
    "flag_avaliacao_valida",
    F.when((F.col("presenca") == 1) & (F.col("preenchimento_caderno") == 1), F.lit(True))
     .otherwise(F.lit(False))
)
 

 
#verificacao de duplicidade
chave_negocio = ["ano", "id_escola", "id_aluno"]
 
total_linhas = df.count()
total_chaves_distintas = df.select(*chave_negocio).distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_avaliacao_alunos] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 
if qtd_duplicados > 0:
    janela = Window.partitionBy(*chave_negocio).orderBy(F.col("proficiencia").desc_nulls_last())
    df = (
        df.withColumn("_rn", F.row_number().over(janela))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )
 
df_municipios_bruto = spark.table(MUNICIPIOS_TABELA)
dh_municipios_recente = (
    df_municipios_bruto
    .agg(F.max("dh_referencia_ingestao_origem").alias("dh_max"))
    .collect()[0]["dh_max"]
)
df_municipios = (
    df_municipios_bruto
    .filter(F.col("dh_referencia_ingestao_origem") == dh_municipios_recente)
    .select("id_municipio")
)

 
#colunas de dat_ref
df = (
    df.withColumn("dh_referencia_ingestao_origem", F.lit(dh_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)
 
#Cria tabela silver
df_final = df.select(
    "ano", "id_municipio", "id_escola", "id_aluno",
    "caderno", "serie", "cd_rede",
    "presenca", "preenchimento_caderno", "flag_avaliacao_valida",
    "alfabetizado", "proficiencia", "peso_aluno",
    "dh_referencia_ingestao_origem", "dh_processamento_silver",
)
 
(
    df_final
    .write
    .format("delta")
    .partitionBy("dh_processamento_silver")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_avaliacao_alunos] carga {dh_mais_recente} adicionada (append) em {SILVER_TABELA}")



In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
 

BRONZE_TABELA = f"workspace.bronze.b_avaliacao_municipio"
SILVER_TABELA = f"workspace.silver.s_avaliacao_municipio"
MUNICIPIOS_TABELA = f"workspace.silver.s_municipios_br"
 
df_bronze = spark.table(BRONZE_TABELA)
 
dt_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dt_max"))
    .collect()[0]["dt_max"]
)
 
df = df_bronze.filter(F.col("dt_ingestao") == dt_mais_recente)
 
print(f"[s_avaliacao_municipio] dt_ingestao utilizada: {dt_mais_recente}")
print(f"[s_avaliacao_municipio] linhas na carga mais recente: {df.count()}")
 
#Padronização das colunas. Mantive o mesmo padrão feito na ultima tabela, mantendo tudo o que é id como string (como no caso do id_municipio)

df = df.select(
    F.col("ano").cast("int").alias("ano"),
    F.lpad(F.col("id_municipio").cast("string"), 7, "0").alias("id_municipio"),
    F.col("serie").cast("int").alias("serie"),
    F.col("rede").cast("int").alias("cd_rede"),
    F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
    F.col("media_portugues").cast("double").alias("media_portugues"),
    F.col("proporcao_aluno_nivel_0").cast("double").alias("proporcao_aluno_nivel_0"),
    F.col("proporcao_aluno_nivel_1").cast("double").alias("proporcao_aluno_nivel_1"),
    F.col("proporcao_aluno_nivel_2").cast("double").alias("proporcao_aluno_nivel_2"),
    F.col("proporcao_aluno_nivel_3").cast("double").alias("proporcao_aluno_nivel_3"),
    F.col("proporcao_aluno_nivel_4").cast("double").alias("proporcao_aluno_nivel_4"),
    F.col("proporcao_aluno_nivel_5").cast("double").alias("proporcao_aluno_nivel_5"),
    F.col("proporcao_aluno_nivel_6").cast("double").alias("proporcao_aluno_nivel_6"),
    F.col("proporcao_aluno_nivel_7").cast("double").alias("proporcao_aluno_nivel_7"),
    F.col("proporcao_aluno_nivel_8").cast("double").alias("proporcao_aluno_nivel_8"),
)
 
#Aqui tem um problema de nulls nas colunas de distribuição do nivel.
#mantive as colunas pois há municipios que não tem essa informação publicada (pode ser que num futuro venham a ter)
#criei uma coluna se caso a coluna de nivel estiver  preenchida
df = df.withColumn(
    "flag_distribuicao_nivel_publicada",
    F.col("proporcao_aluno_nivel_0").isNotNull()
)
 

# Verificação de duplicidades
chave_negocio = ["ano", "id_municipio", "serie", "cd_rede"]
 
total_linhas = df.count()
total_chaves_distintas = df.select(*chave_negocio).distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_avaliacao_municipio] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 
if qtd_duplicados > 0:
    colunas_conteudo = [c for c in df.columns if c != "dt_ingestao"]
    df_hash = df.withColumn("_hash_conteudo", F.sha2(F.concat_ws("||", *colunas_conteudo), 256))
 
    qtd_hashes_por_chave = (
        df_hash.groupBy(*chave_negocio)
        .agg(F.countDistinct("_hash_conteudo").alias("qtd_hashes_distintos"))
    )
 
    chaves_com_divergencia = qtd_hashes_por_chave.filter(F.col("qtd_hashes_distintos") > 1).count()
    print(f"[s_avaliacao_municipio] chaves com CONTEÚDO DIVERGENTE entre cargas do mesmo dia: {chaves_com_divergencia}")
 
    if chaves_com_divergencia > 0:
        raise ValueError(
            f"{chaves_com_divergencia} registros de município têm conteúdo diferente entre "
            "cargas do mesmo dia (dt_ingestao com granularidade de dia não distingue "
            "execuções). Investigar a ingestão da Bronze antes de prosseguir."
        )
 
    janela = Window.partitionBy(*chave_negocio).orderBy(F.lit(1))
    df = (
        df.withColumn("_rn", F.row_number().over(janela))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )
    print(f"[s_avaliacao_municipio] duplicados eram cópias idênticas (reprocessamento) — deduplicados com segurança.")
 
df_municipios_bruto = spark.table(MUNICIPIOS_TABELA)
dh_municipios_recente = (
    df_municipios_bruto
    .agg(F.max("dh_referencia_ingestao_origem").alias("dh_max"))
    .collect()[0]["dh_max"]
)
df_municipios = (
    df_municipios_bruto
    .filter(F.col("dh_referencia_ingestao_origem") == dh_municipios_recente)
    .select("id_municipio")
)
 
qtd_municipio_invalido = df.join(df_municipios, on="id_municipio", how="left_anti").count()
print(f"[s_avaliacao_municipio] registros com id_municipio não encontrado na dimensão: {qtd_municipio_invalido}")
 
if qtd_municipio_invalido > 0:
    df_quarentena = df.join(df_municipios, on="id_municipio", how="left_anti")
    (
        df_quarentena
        .withColumn("dt_ingestao_origem", F.lit(dt_mais_recente))
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"workspace.silver.s_avaliacao_municipio_quarentena")
    )
    df = df.join(df_municipios, on="id_municipio", how="left_semi")
 
# verificando se tem algum valor de taxa fora do alvo
qtd_taxa_fora_faixa = df.filter(
    F.col("taxa_alfabetizacao").isNotNull()
    & ((F.col("taxa_alfabetizacao") < 0) | (F.col("taxa_alfabetizacao") > 100))
).count()
print(f"[s_avaliacao_municipio] registros com taxa_alfabetizacao fora da faixa 0-100: {qtd_taxa_fora_faixa}")
 
# data_ref
df = (
    df.withColumn("dt_ingestao_origem", F.lit(dt_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)
 
#salvar na tabela silver
df_final = df.select(
    "ano", "id_municipio", "serie", "cd_rede",
    "taxa_alfabetizacao", "media_portugues",
    "flag_distribuicao_nivel_publicada",
    "proporcao_aluno_nivel_0", "proporcao_aluno_nivel_1", "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3", "proporcao_aluno_nivel_4", "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6", "proporcao_aluno_nivel_7", "proporcao_aluno_nivel_8",
    "dt_ingestao_origem", "dh_processamento_silver",
)
 
(
    df_final
    .write
    .format("delta")
    .partitionBy("dh_processamento_silver")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_avaliacao_municipio] carga {dt_mais_recente} adicionada (append) em {SILVER_TABELA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
 

BRONZE_TABELA = f"workspace.bronze.b_avaliacao_uf"
SILVER_TABELA = f"workspace.silver.s_avaliacao_uf"
MUNICIPIOS_TABELA = f"workspace.silver.s_municipios_br"  
 
#mais recente da bronze da carga
df_bronze = spark.table(BRONZE_TABELA)
 
dt_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dt_max"))
    .collect()[0]["dt_max"]
)
 
df = df_bronze.filter(F.col("dt_ingestao") == dt_mais_recente)
 
print(f"[s_avaliacao_uf] dt_ingestao utilizada: {dt_mais_recente}")
print(f"[s_avaliacao_uf] linhas na carga mais recente: {df.count()}")
 
# padronizando  as colunas
df = df.select(
    F.col("ano").cast("int").alias("ano"),
    F.upper(F.trim(F.col("sigla_uf"))).alias("sigla_uf"),
    F.col("serie").cast("int").alias("serie"),
    F.col("rede").cast("int").alias("cd_rede"),
    F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
    F.col("media_portugues").cast("double").alias("media_portugues"),
    F.col("proporcao_aluno_nivel_0").cast("double").alias("proporcao_aluno_nivel_0"),
    F.col("proporcao_aluno_nivel_1").cast("double").alias("proporcao_aluno_nivel_1"),
    F.col("proporcao_aluno_nivel_2").cast("double").alias("proporcao_aluno_nivel_2"),
    F.col("proporcao_aluno_nivel_3").cast("double").alias("proporcao_aluno_nivel_3"),
    F.col("proporcao_aluno_nivel_4").cast("double").alias("proporcao_aluno_nivel_4"),
    F.col("proporcao_aluno_nivel_5").cast("double").alias("proporcao_aluno_nivel_5"),
    F.col("proporcao_aluno_nivel_6").cast("double").alias("proporcao_aluno_nivel_6"),
    F.col("proporcao_aluno_nivel_7").cast("double").alias("proporcao_aluno_nivel_7"),
    F.col("proporcao_aluno_nivel_8").cast("double").alias("proporcao_aluno_nivel_8"),
)
 
#Aqui vou manter aquela flag que fizemos no notebook anterior
df = df.withColumn(
    "flag_distribuicao_nivel_publicada",
    F.col("proporcao_aluno_nivel_0").isNotNull()
)
 
#essa chave aqui é o que deveria ser particionado
chave_negocio = ["ano", "sigla_uf", "serie", "cd_rede"]
 
total_linhas = df.count()
total_chaves_distintas = df.select(*chave_negocio).distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_avaliacao_uf] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 
if qtd_duplicados > 0:
    colunas_conteudo = [c for c in df.columns if c != "dt_ingestao"]
    df_hash = df.withColumn("_hash_conteudo", F.sha2(F.concat_ws("||", *colunas_conteudo), 256))
 
    qtd_hashes_por_chave = (
        df_hash.groupBy(*chave_negocio)
        .agg(F.countDistinct("_hash_conteudo").alias("qtd_hashes_distintos"))
    )
 
    chaves_com_divergencia = qtd_hashes_por_chave.filter(F.col("qtd_hashes_distintos") > 1).count()
    print(f"[s_avaliacao_uf] chaves com CONTEÚDO DIVERGENTE entre cargas do mesmo dia: {chaves_com_divergencia}")
 
    if chaves_com_divergencia > 0:
        raise ValueError(
            f"{chaves_com_divergencia} registros de UF têm conteúdo diferente entre "
            "cargas do mesmo dia (dt_ingestao com granularidade de dia não distingue "
            "execuções). Investigar a ingestão da Bronze antes de prosseguir."
        )
 
    janela = Window.partitionBy(*chave_negocio).orderBy(F.lit(1))
    df = (
        df.withColumn("_rn", F.row_number().over(janela))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )
    print(f"[s_avaliacao_uf] duplicados eram cópias idênticas (reprocessamento) — deduplicados com segurança.")
 

 

# Verificação de valores null
qtd_taxa_fora_faixa = df.filter(
    F.col("taxa_alfabetizacao").isNotNull()
    & ((F.col("taxa_alfabetizacao") < 0) | (F.col("taxa_alfabetizacao") > 100))
).count()
print(f"[s_avaliacao_uf] registros com taxa_alfabetizacao fora da faixa 0-100: {qtd_taxa_fora_faixa}")
 
# criação da dat_ref
df = (
    df.withColumn("dt_ingestao_origem", F.lit(dt_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)
 
# escrita da tabela na silver
df_final = df.select(
    "ano", "sigla_uf", "serie", "cd_rede",
    "taxa_alfabetizacao", "media_portugues",
    "flag_distribuicao_nivel_publicada",
    "proporcao_aluno_nivel_0", "proporcao_aluno_nivel_1", "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3", "proporcao_aluno_nivel_4", "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6", "proporcao_aluno_nivel_7", "proporcao_aluno_nivel_8",
    "dt_ingestao_origem", "dh_processamento_silver",
)
 
(
    df_final
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_avaliacao_uf] carga {dt_mais_recente} adicionada (append) em {SILVER_TABELA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
 
BRONZE_TABELA = f"workspace.bronze.b_meta_alfabetizacao_brasil"
SILVER_TABELA = f"workspace.silver.s_meta_alfabetizacao_brasil"
 
#carga mais recente
df_bronze = spark.table(BRONZE_TABELA)
 
dt_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dt_max"))
    .collect()[0]["dt_max"]
)
 
df = df_bronze.filter(F.col("dt_ingestao") == dt_mais_recente)
 
print(f"[s_meta_alfabetizacao_brasil] dt_ingestao utilizada: {dt_mais_recente}")
print(f"[s_meta_alfabetizacao_brasil] linhas na carga mais recente: {df.count()}")
 
# Padronização das clunas
df = df.select(
    F.col("ano").cast("int").alias("ano"),
    F.trim(F.col("rede")).alias("rede"),
    F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
    F.col("meta_alfabetizacao_2024").cast("double").alias("meta_2024"),
    F.col("meta_alfabetizacao_2025").cast("double").alias("meta_2025"),
    F.col("meta_alfabetizacao_2026").cast("double").alias("meta_2026"),
    F.col("meta_alfabetizacao_2027").cast("double").alias("meta_2027"),
    F.col("meta_alfabetizacao_2028").cast("double").alias("meta_2028"),
    F.col("meta_alfabetizacao_2029").cast("double").alias("meta_2029"),
    F.col("meta_alfabetizacao_2030").cast("double").alias("meta_2030"),
    F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
)
 
# Verificação de duplicadas por ano
chave_negocio = ["ano"]
 
total_linhas = df.count()
total_chaves_distintas = df.select(*chave_negocio).distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_meta_alfabetizacao_brasil] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 

 
# Verificação se as taxas estão entre 0 e 100
colunas_percentual = [
    "taxa_alfabetizacao", "meta_2024", "meta_2025", "meta_2026",
    "meta_2027", "meta_2028", "meta_2029", "meta_2030", "percentual_participacao",
]
for coluna in colunas_percentual:
    qtd_fora_faixa = df.filter(
        F.col(coluna).isNotNull() & ((F.col(coluna) < 0) | (F.col(coluna) > 100))
    ).count()
    if qtd_fora_faixa > 0:
        print(f"[s_meta_alfabetizacao_brasil] {coluna}: {qtd_fora_faixa} registros fora da faixa 0-100.")
 
#criação da dat_ref
df = (
    df.withColumn("dt_ingestao_origem", F.lit(dt_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)
 
# carga na silver
df_final = df.select(
    "ano", "rede", "taxa_alfabetizacao",
    "meta_2024", "meta_2025", "meta_2026", "meta_2027",
    "meta_2028", "meta_2029", "meta_2030",
    "percentual_participacao",
    "dt_ingestao_origem", "dh_processamento_silver",
)
 
(
    df_final
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_meta_alfabetizacao_brasil] carga {dt_mais_recente} adicionada (append) em {SILVER_TABELA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
 

BRONZE_TABELA = f"workspace.bronze.b_meta_alfabetizacao_uf"
SILVER_TABELA = f"workspace.silver.s_meta_alfabetizacao_uf"
MUNICIPIOS_TABELA = f"workspace.silver.s_municipios_br"  # fonte da lista válida de sigla_uf
 
#carga mais recente
df_bronze = spark.table(BRONZE_TABELA)
 
dt_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dt_max"))
    .collect()[0]["dt_max"]
)
 
df = df_bronze.filter(F.col("dt_ingestao") == dt_mais_recente)
 
print(f"[s_meta_alfabetizacao_uf] dt_ingestao utilizada: {dt_mais_recente}")
print(f"[s_meta_alfabetizacao_uf] linhas na carga mais recente: {df.count()}")
 
# Padronização das colunas
df = df.select(
    F.col("ano").cast("int").alias("ano"),
    F.upper(F.trim(F.col("sigla_uf"))).alias("sigla_uf"),
    F.trim(F.col("rede")).alias("rede"),
    F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
    F.col("meta_alfabetizacao_2024").cast("double").alias("meta_2024"),
    F.col("meta_alfabetizacao_2025").cast("double").alias("meta_2025"),
    F.col("meta_alfabetizacao_2026").cast("double").alias("meta_2026"),
    F.col("meta_alfabetizacao_2027").cast("double").alias("meta_2027"),
    F.col("meta_alfabetizacao_2028").cast("double").alias("meta_2028"),
    F.col("meta_alfabetizacao_2029").cast("double").alias("meta_2029"),
    F.col("meta_alfabetizacao_2030").cast("double").alias("meta_2030"),
    F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
)
 

# Verificacao de duplicidades
chave_negocio = ["ano", "sigla_uf"]
 
total_linhas = df.count()
total_chaves_distintas = df.select(*chave_negocio).distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_meta_alfabetizacao_uf] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 
if qtd_duplicados > 0:
    colunas_conteudo = [c for c in df.columns if c != "dt_ingestao"]
    df_hash = df.withColumn("_hash_conteudo", F.sha2(F.concat_ws("||", *colunas_conteudo), 256))
 
    qtd_hashes_por_chave = (
        df_hash.groupBy(*chave_negocio)
        .agg(F.countDistinct("_hash_conteudo").alias("qtd_hashes_distintos"))
    )
 
    chaves_com_divergencia = qtd_hashes_por_chave.filter(F.col("qtd_hashes_distintos") > 1).count()
    print(f"[s_meta_alfabetizacao_uf] chaves com CONTEÚDO DIVERGENTE entre cargas do mesmo dia: {chaves_com_divergencia}")
 
    if chaves_com_divergencia > 0:
        raise ValueError(
            f"{chaves_com_divergencia} registros de meta UF têm conteúdo diferente entre "
            "cargas do mesmo dia. Investigar a ingestão da Bronze antes de prosseguir."
        )
 
    janela = Window.partitionBy(*chave_negocio).orderBy(F.lit(1))
    df = (
        df.withColumn("_rn", F.row_number().over(janela))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )
    print(f"[s_meta_alfabetizacao_uf] duplicados eram cópias idênticas (reprocessamento) — deduplicados com segurança.")
 

# problema de ufs vazias, já sabemos que RR não tem informações
qtd_uf_com_meta_totalmente_nula = df.filter(
    F.col("taxa_alfabetizacao").isNull() & F.col("meta_2030").isNull()
).count()
print(f"[s_meta_alfabetizacao_uf] registros com taxa e meta totalmente nulas (ex.: RR): {qtd_uf_com_meta_totalmente_nula}")
 
# Verificação de valores inconcistantes
colunas_percentual = [
    "taxa_alfabetizacao", "meta_2024", "meta_2025", "meta_2026",
    "meta_2027", "meta_2028", "meta_2029", "meta_2030", "percentual_participacao",
]
for coluna in colunas_percentual:
    qtd_fora_faixa = df.filter(
        F.col(coluna).isNotNull() & ((F.col(coluna) < 0) | (F.col(coluna) > 100))
    ).count()
    if qtd_fora_faixa > 0:
        print(f"[s_meta_alfabetizacao_uf] {coluna}: {qtd_fora_faixa} registros fora da faixa 0-100.")
 
# criação da dat_ref
df = (
    df.withColumn("dt_ingestao_origem", F.lit(dt_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)
 
# carga na silver
df_final = df.select(
    "ano", "sigla_uf", "rede", "taxa_alfabetizacao",
    "meta_2024", "meta_2025", "meta_2026", "meta_2027",
    "meta_2028", "meta_2029", "meta_2030",
    "percentual_participacao",
    "dt_ingestao_origem", "dh_processamento_silver",
)
 
(
    df_final
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_meta_alfabetizacao_uf] carga {dt_mais_recente} adicionada (append) em {SILVER_TABELA}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

BRONZE_TABELA = f"workspace.bronze.b_meta_alfabetizacao_municipio"
SILVER_TABELA = f"workspace.silver.s_meta_alfabetizacao_municipio"
MUNICIPIOS_TABELA = f"workspace.silver.s_municipios_br"
 
#carga mais recente
df_bronze = spark.table(BRONZE_TABELA)
 
dt_mais_recente = (
    df_bronze
    .agg(F.max("dt_ingestao").alias("dt_max"))
    .collect()[0]["dt_max"]
)
 
df = df_bronze.filter(F.col("dt_ingestao") == dt_mais_recente)
 
print(f"[s_meta_alfabetizacao_municipio] dt_ingestao utilizada: {dt_mais_recente}")
print(f"[s_meta_alfabetizacao_municipio] linhas na carga mais recente: {df.count()}")
 
# Padronização das colunas
df = df.select(
    F.col("ano").cast("int").alias("ano"),
    F.lpad(F.col("id_municipio").cast("string"), 7, "0").alias("id_municipio"),
    F.trim(F.col("rede")).alias("rede"),
    F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
    F.col("meta_alfabetizacao_2024").cast("double").alias("meta_2024"),
    F.col("meta_alfabetizacao_2025").cast("double").alias("meta_2025"),
    F.col("meta_alfabetizacao_2026").cast("double").alias("meta_2026"),
    F.col("meta_alfabetizacao_2027").cast("double").alias("meta_2027"),
    F.col("meta_alfabetizacao_2028").cast("double").alias("meta_2028"),
    F.col("meta_alfabetizacao_2029").cast("double").alias("meta_2029"),
    F.col("meta_alfabetizacao_2030").cast("double").alias("meta_2030"),
    F.col("nivel_alfabetizacao").cast("int").alias("nivel_alfabetizacao"),
    F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
)
 

# verificação de duplicadas
chave_negocio = ["ano", "id_municipio"]
 
total_linhas = df.count()
total_chaves_distintas = df.select(*chave_negocio).distinct().count()
qtd_duplicados = total_linhas - total_chaves_distintas
 
print(f"[s_meta_alfabetizacao_municipio] total linhas: {total_linhas} | chaves distintas: {total_chaves_distintas} | duplicados: {qtd_duplicados}")
 


 
# verificação dos valores null
qtd_taxa_nula = df.filter(F.col("taxa_alfabetizacao").isNull()).count()
qtd_meta_2024_nula = df.filter(F.col("meta_2024").isNull()).count()
qtd_nivel_nulo = df.filter(F.col("nivel_alfabetizacao").isNull()).count()
print(f"[s_meta_alfabetizacao_municipio] taxa_alfabetizacao nula: {qtd_taxa_nula} | meta_2024 nula: {qtd_meta_2024_nula} | nivel_alfabetizacao nulo: {qtd_nivel_nulo}")
 
# validação da consistencia dos numeros
colunas_percentual = [
    "taxa_alfabetizacao", "meta_2024", "meta_2025", "meta_2026",
    "meta_2027", "meta_2028", "meta_2029", "meta_2030", "percentual_participacao",
]
for coluna in colunas_percentual:
    qtd_fora_faixa = df.filter(
        F.col(coluna).isNotNull() & ((F.col(coluna) < 0) | (F.col(coluna) > 100))
    ).count()
    if qtd_fora_faixa > 0:
        print(f"[s_meta_alfabetizacao_municipio] {coluna}: {qtd_fora_faixa} registros fora da faixa 0-100.")
 
# nivel_alfabetizacao é categórico (0 a 5, segundo divulgação do INEP) —
# qualquer valor fora dessa faixa é erro de fonte.
qtd_nivel_invalido = df.filter(
    F.col("nivel_alfabetizacao").isNotNull()
    & ((F.col("nivel_alfabetizacao") < 0) | (F.col("nivel_alfabetizacao") > 5))
).count()
if qtd_nivel_invalido > 0:
    print(f"[s_meta_alfabetizacao_municipio] nivel_alfabetizacao fora da faixa 0-5: {qtd_nivel_invalido} registros.")
 
# criação da dat_ref
df = (
    df.withColumn("dt_ingestao_origem", F.lit(dt_mais_recente))
      .withColumn("dh_processamento_silver", F.current_timestamp())
)
 
# carga na silver
df_final = df.select(
    "ano", "id_municipio", "rede", "taxa_alfabetizacao",
    "meta_2024", "meta_2025", "meta_2026", "meta_2027",
    "meta_2028", "meta_2029", "meta_2030",
    "nivel_alfabetizacao", "percentual_participacao",
    "dt_ingestao_origem", "dh_processamento_silver",
)
 
(
    df_final
    .write
    .format("delta")
    .partitionBy("ano")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TABELA)
)
 
print(f"[s_meta_alfabetizacao_municipio] carga {dt_mais_recente} adicionada (append) em {SILVER_TABELA}")